# Section 9. 근거가 추적되는 Vector RAG

공개 실습용 notebook입니다. 외부 API 없이 RAG 단계와 검증 정책을
재현합니다. `retrieve`는 Section 8의 vector retriever와 같은 입출력 계약을 가지는
결정적 제작용 대역입니다. 실제 수업에서는 이 함수만 검증된 vector retriever로 교체합니다.

## API key 설정 — 실제 API 선택 실습에서만 사용

이 Section의 기본 실습은 API key 없이 실행됩니다. 실제 API gate까지 실행하려면 아래 셀에 수업용 key를 입력하고 `RUN_LIVE_API = True`로 바꿉니다. key가 들어간 notebook은 공유하거나 제출하지 않습니다.


In [ ]:
import os

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "여기에_수업용_API_KEY를_붙여넣으세요")
RUN_LIVE_API = os.getenv("RUN_LIVE_API", "0") == "1"  # 직접 실행할 때는 True로 변경

if OPENAI_API_KEY and not OPENAI_API_KEY.startswith("여기에_"):
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    os.environ["RUN_LIVE_API"] = "1" if RUN_LIVE_API else "0"
    print("API key가 현재 notebook 세션에 설정되었습니다.")
    print("실제 API gate 실행:", RUN_LIVE_API)
else:
    os.environ["RUN_LIVE_API"] = "0"
    print("API key가 없어 기본 offline 실습만 실행합니다.")


In [ ]:
from __future__ import annotations

import json
import os
import re
from pathlib import Path
from typing import Literal

from pydantic import BaseModel, Field


def find_data(name: str) -> Path:
    candidates = [
        Path("../data") / name,
        Path(__file__).resolve().parents[1] / "data" / name
        if "__file__" in globals() else Path("__missing__"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(name)


CHUNKS = json.loads(find_data("section09_policy_chunks.json").read_text(encoding="utf-8"))
CHUNK_BY_ID = {chunk["chunk_id"]: chunk for chunk in CHUNKS}
assert len(CHUNK_BY_ID) == len(CHUNKS)

## 데이터 계약: 단계의 경계에서 무엇을 기록하는가

In [ ]:
class RetrievedChunk(BaseModel):
    source_id: str
    chunk_id: str
    text: str
    score: float


class Citation(BaseModel):
    chunk_id: str
    quote: str = Field(min_length=1)


class RagAnswer(BaseModel):
    status: Literal["answered", "not_answered", "needs_review"]
    answer: str
    citations: list[Citation] = Field(default_factory=list)


class RagTrace(BaseModel):
    original_question: str
    retrieval_query: str
    retrieved: list[RetrievedChunk]
    result: RagAnswer
    validation_errors: list[str]

## 제작용 결정적 retriever

아래 점수는 vector similarity가 아닙니다. API 없이 파이프라인 계약과 실패 처리를
시험하기 위한 keyword 대역입니다. 실제 vector score와 동일하게 해석하면 안 됩니다.

In [ ]:
KEYWORDS = {
    "api": ["API", "key", "인증정보", "비밀번호", "보관", "저장"],
    "leak": ["유출", "노출", "의심", "폐기", "발급", "대응"],
    "data": ["개인정보", "비공개", "연구자료", "학습", "예제", "자료"],
    "meeting": ["미팅", "요일", "시각", "몇 시", "월요일"],
    "submission": ["실습", "결과", "설정", "성공", "실패", "가설", "포함"],
}


def features(text: str) -> set[str]:
    lowered = text.lower()
    return {
        label for label, words in KEYWORDS.items()
        if any(word.lower() in lowered for word in words)
    }


def retrieve(query: str, k: int = 2) -> list[RetrievedChunk]:
    query_features = features(query)
    ranked = []
    for chunk in CHUNKS:
        overlap = query_features & features(chunk["text"])
        score = len(overlap) / max(1, len(query_features))
        if score > 0:
            ranked.append(RetrievedChunk(**chunk, score=score))
    ranked.sort(key=lambda item: (-item.score, item.chunk_id))
    return ranked[:k]

## 대화 기록과 검색 질문 분리

In [ ]:
def build_retrieval_query(question: str, prior_user_question: str | None = None) -> str:
    # 실제 서비스에서는 별도 rewrite model을 평가해야 합니다. 여기서는 대명사 질문에만
    # 직전 사용자 질문을 붙여 재현 가능한 최소 동작을 관찰합니다.
    if prior_user_question and re.search(r"그것|그 경우|그때", question):
        return f"{prior_user_question} / 후속 질문: {question}"
    return question

## 제작용 generator와 답 없음 정책

In [ ]:
def generate(question: str, retrieved: list[RetrievedChunk]) -> RagAnswer:
    q = question.lower()
    if "몇 시" in q or "휴가" in q:
        return RagAnswer(
            status="not_answered",
            answer="검색된 근거만으로 답할 수 없습니다.",
        )
    if "보관" in q and any(c.chunk_id == "security-guide:000" for c in retrieved):
        quote = "인증정보는 승인된 환경변수 또는 비밀 저장소에서 불러옵니다."
        return RagAnswer(status="answered", answer="API key는 승인된 환경변수 또는 비밀 저장소에서 불러옵니다.", citations=[Citation(chunk_id="security-guide:000", quote=quote)])
    if ("유출" in q or "노출" in q) and any(c.chunk_id == "security-guide:001" for c in retrieved):
        quote = "해당 key를 즉시 폐기하고 새 key를 발급합니다."
        return RagAnswer(status="answered", answer="노출된 key를 폐기하고 새 key를 발급한 뒤 범위를 확인해 담당자에게 알립니다.", citations=[Citation(chunk_id="security-guide:001", quote=quote)])
    if "비공개" in q and any(c.chunk_id == "data-guide:000" for c in retrieved):
        quote = "학습 예제에는 개인정보나 비공개 연구자료를 넣지 않습니다."
        return RagAnswer(status="answered", answer="비공개 연구자료는 학습 예제에 넣지 않습니다.", citations=[Citation(chunk_id="data-guide:000", quote=quote)])
    if "실습" in q and any(c.chunk_id == "submission-guide:000" for c in retrieved):
        quote = "사용한 설정, 성공 사례, 실패 사례, 다음에 확인할 가설을 포함합니다."
        return RagAnswer(status="answered", answer="설정, 성공 사례, 실패 사례와 다음 가설을 포함합니다.", citations=[Citation(chunk_id="submission-guide:000", quote=quote)])
    return RagAnswer(status="needs_review", answer="관련 문서는 찾았지만 근거가 질문에 충분한지 검토가 필요합니다.")


def validate(answer: RagAnswer, retrieved: list[RetrievedChunk]) -> list[str]:
    errors: list[str] = []
    retrieved_by_id = {item.chunk_id: item for item in retrieved}
    if answer.status == "answered" and not answer.citations:
        errors.append("answered 상태에는 citation이 필요합니다.")
    if answer.status != "answered" and answer.citations:
        errors.append("미답변/검토 상태에는 citation을 붙이지 않습니다.")
    for citation in answer.citations:
        chunk = retrieved_by_id.get(citation.chunk_id)
        if chunk is None:
            errors.append(f"검색되지 않은 chunk 인용: {citation.chunk_id}")
        elif citation.quote not in chunk.text:
            errors.append(f"원문에 없는 quote: {citation.chunk_id}")
    return errors


def run_rag(question: str, prior_user_question: str | None = None) -> RagTrace:
    retrieval_query = build_retrieval_query(question, prior_user_question)
    retrieved = retrieve(retrieval_query)
    result = generate(question, retrieved)
    return RagTrace(
        original_question=question,
        retrieval_query=retrieval_query,
        retrieved=retrieved,
        result=result,
        validation_errors=validate(result, retrieved),
    )

## 정답 가능·불가능·검토 필요 사례 실행

In [ ]:
QUESTIONS = [
    "API key는 어디에 보관해야 하나요?",
    "인증정보 유출이 의심될 때 첫 대응은 무엇인가요?",
    "정기 미팅은 매주 월요일 몇 시인가요?",
    "보안과 관련해 무엇을 준비해야 하나요?",
]

for question in QUESTIONS:
    trace = run_rag(question)
    print("\nQUESTION:", question)
    print("RETRIEVED:", [item.chunk_id for item in trace.retrieved])
    print("STATUS:", trace.result.status)
    print("ANSWER:", trace.result.answer)
    print("VALIDATION:", trace.validation_errors or "PASS")

assert run_rag(QUESTIONS[0]).result.status == "answered"
assert run_rag(QUESTIONS[2]).result.status == "not_answered"
assert run_rag(QUESTIONS[3]).result.status == "needs_review"
assert all(not run_rag(question).validation_errors for question in QUESTIONS)

## 인용 검사가 잡아내는 실패

In [ ]:
retrieved = retrieve("API key는 어디에 보관해야 하나요?")
broken = RagAnswer(
    status="answered",
    answer="API key는 개인 메모장에 둡니다.",
    citations=[Citation(chunk_id="security-guide:000", quote="개인 메모장에 둡니다.")],
)
broken_errors = validate(broken, retrieved)
print("의도한 citation failure:", broken_errors)
assert broken_errors

## 후속 질문의 검색 query 확인

In [ ]:
follow_up = run_rag("그 경우 가장 먼저 무엇을 하나요?", prior_user_question="인증정보 유출이 의심됩니다.")
print("원 질문:", follow_up.original_question)
print("검색 질문:", follow_up.retrieval_query)
assert follow_up.original_question != follow_up.retrieval_query

## 선택 실습: 실제 생성 모델 end-to-end readiness gate

위 assertion은 파이프라인 **계약**을 오프라인에서 검증합니다. 아래 gate는 별개의
end-to-end 검사입니다. `RUN_LIVE_API=1`과 `OPENAI_API_KEY`가 모두 있을 때만 정확히
세 질문을 각 1회 호출하며, 그 외에는 호출하지 않고 `NOT COMPLETED`를 출력합니다.

In [ ]:
LIVE_MODEL = "gpt-5.4-mini"
LIVE_CASES = [
    ("API key는 어디에 보관해야 하나요?", None, "answered"),
    ("정기 미팅은 매주 월요일 몇 시인가요?", None, "not_answered"),
    # 검색 query에는 보안 주제가 남아 있지만 원 질문의 대명사만으로는 보관과 유출 대응 중
    # 어느 절차를 묻는지 정할 수 없습니다. 관련 검색 결과와 답변 가능성을 분리하는 사례입니다.
    ("그 경우에는 무엇을 해야 하나요?", "API key 인증정보 보안", "needs_review"),
]
MAX_LIVE_CALLS = len(LIVE_CASES)
MAX_OUTPUT_TOKENS_PER_CALL = 800
INPUT_USD_PER_MILLION = 0.75
OUTPUT_USD_PER_MILLION = 4.50


def context_for(items: list[RetrievedChunk]) -> str:
    if not items:
        return "(검색된 근거 없음)"
    return "\n\n".join(
        f"[chunk_id={item.chunk_id}]\n{item.text}" for item in items
    )


def estimate_live_budget() -> tuple[int, float]:
    # API tokenizer를 호출하지 않는 실행 전 보수적 근사치입니다. 한글은 문자당 1 token로
    # 과대 추정하고, 출력은 설정한 최대치 전체를 사용한다고 가정합니다.
    estimated_input_tokens = 0
    for question, retrieval_query, _ in LIVE_CASES:
        context = context_for(retrieve(retrieval_query or question))
        estimated_input_tokens += len(question) + len(context) + 1_200
    estimated_output_tokens = MAX_LIVE_CALLS * MAX_OUTPUT_TOKENS_PER_CALL
    usd = (
        estimated_input_tokens * INPUT_USD_PER_MILLION
        + estimated_output_tokens * OUTPUT_USD_PER_MILLION
    ) / 1_000_000
    return estimated_input_tokens, usd


def run_live_readiness_gate() -> None:
    estimated_input_tokens, estimated_usd = estimate_live_budget()
    print("\nLIVE READINESS GATE")
    print("model:", LIVE_MODEL)
    print("call count:", MAX_LIVE_CALLS, "(재시도 없음)")
    print("estimated maximum budget: $%.4f" % estimated_usd)
    print("pricing basis: $0.75/M input, $4.50/M output (2026-07-14; verify official pricing before use)")

    if os.getenv("RUN_LIVE_API") != "1":
        print("LIVE GATE: NOT COMPLETED — RUN_LIVE_API=1이 아니므로 비용 없는 오프라인 실행만 완료했습니다.")
        return
    if not os.getenv("OPENAI_API_KEY"):
        print("LIVE GATE: NOT COMPLETED — OPENAI_API_KEY 환경변수가 없습니다.")
        return

    from openai import OpenAI

    client = OpenAI()
    total_input = total_output = 0
    for call_number, (question, retrieval_query, expected_status) in enumerate(LIVE_CASES, start=1):
        retrieved = retrieve(retrieval_query or question)
        response = client.responses.parse(
            model=LIVE_MODEL,
            instructions=(
                "당신은 제공된 context만 사용하는 RAG 생성기입니다. 직접 근거가 있으면 answered, "
                "근거가 없으면 not_answered, 관련 내용은 있으나 질문이 넓거나 모호하면 needs_review를 "
                "선택하세요. 특히 서로 다른 절차를 설명하는 여러 chunk가 관련되는데 질문이 어느 절차를 "
                "원하는지 특정하지 않으면, 임의로 범위를 정해 답하지 말고 needs_review를 선택하세요. "
                "answered일 때만 citations를 만들고, not_answered 또는 needs_review이면 citations는 반드시 "
                "빈 배열로 두세요. answered의 quote는 context의 한 chunk에서 글자까지 "
                "정확히 연속 복사하세요. context에 없는 사실을 추측하지 마세요."
            ),
            input=f"질문:\n{question}\n\n허용된 context:\n{context_for(retrieved)}",
            text_format=RagAnswer,
            max_output_tokens=MAX_OUTPUT_TOKENS_PER_CALL,
            store=False,
        )
        result = response.output_parsed
        if result is None:
            raise RuntimeError(f"call {call_number}: 구조화 출력이 반환되지 않았습니다.")
        errors = validate(result, retrieved)
        if result.status != expected_status:
            errors.append(f"예상 상태 {expected_status}, 실제 상태 {result.status}")
        if errors:
            raise AssertionError(f"call {call_number} 검증 실패: {errors}")
        usage = response.usage
        if usage is not None:
            total_input += usage.input_tokens
            total_output += usage.output_tokens
        print(f"call {call_number}/{MAX_LIVE_CALLS}: {question} -> {result.status}, citation validation PASS")

    actual_usd = (
        total_input * INPUT_USD_PER_MILLION + total_output * OUTPUT_USD_PER_MILLION
    ) / 1_000_000
    print(f"usage: input={total_input}, output={total_output}, estimated cost=${actual_usd:.6f}")
    print("LIVE GATE: COMPLETED — schema, 상태 정책, exact citation quote를 실제 응답으로 검증했습니다.")


run_live_readiness_gate()

## 정리 질문

1. retrieval 실패와 generation 실패는 trace의 어느 필드로 구분할 수 있나요?
2. JSON schema를 통과한 답도 왜 citation validation이 필요한가요?
3. 답 없음과 사람 검토를 어떤 기준으로 나누었나요?